In [ ]:
import numpy as np
# import pandas as pd
import matplotlib.pyplot as plt
# import seaborn as sns
# from ripser import ripser
# from persim import PersistenceImager, plot_persistence_diagrams
# from persim.persistent_entropy import persistent_entropy
# from gtda.diagrams import BettiCurve
from tda_methods import GeometryConverter, PersistenceAnalysis, make_timedelay_embeddings, plot_3d_points
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn


In [ ]:
"Torus/Sphere data"

rng     = np.random.default_rng()
start_point, end_point, n_points = 0, 2*np.pi, 1000
u_angle = rng.uniform(start_point, end_point, n_points)
v_angle = rng.uniform(start_point, end_point, n_points)

# u, v = np.meshgrid(u, v)

R_major   = 1
r_tube    = R_major/4
converter = GeometryConverter(R_major, r_tube)
x, y, z   = converter.convert_angles_to_torus_xyz(u_angle, v_angle)
# # x, y, z   = converter.convert_angles_to_sphere_xyz(u_angle, v_angle)
noise = rng.normal(0, 0.04, n_points)
x += noise  # Add some noise to v
y += noise  # Add some noise to u
z += noise  # Add some noise to both
geo_coordinates = np.column_stack((x, y, z))  # shape (N, 3)

# x1,y1,z1,x2,y2,z2 = converter.convert_angles_to_2torus_xyz(u_angle, v_angle, d_shift=R_major)
# geo_coordinates   = np.vstack([np.column_stack((x1,y1,z1)), np.column_stack((x2,y2,z2))])
plot_3d_points((geo_coordinates[:, 0], geo_coordinates[:, 1], geo_coordinates[:, 2]))


In [ ]:
"persistence"

persistence    = PersistenceAnalysis(max_dim=1)
# diagrams_list  = persistence.plot_persistence_diagrams(geo_coordinates, to_plot=True)
diagrams_list  = persistence.compute_persistence_diagrams(geo_coordinates)
diagrams_clean = persistence.remove_inf(diagrams_list)
entropy_array  = persistence.compute_entropy(diagrams_clean)
persistence_images_list = persistence.compute_persistence_image(diagrams_clean)
betti_curves_array      = persistence.compute_betti_curves(diagrams_clean)

print("Persistence array:", entropy_array)
print("Betti curves shape: (batch, homology_dim, filtration_steps) =", betti_curves_array.shape)

persistence.plot_persistence_diagrams(diagrams_list)
persistence.plot_entropy(entropy_array)
persistence.plot_persistence_image(persistence_images_list[1])  # H1
persistence.plot_betti_curves(betti_curves_array)


In [ ]:
"timedelay embeddings"

x  = np.linspace(0, 14*np.pi, 1000)
y1 = (0.5+0.5*x) * np.sin(2*x + 1) + 0.04 * rng.normal(size=x.shape)
y2 = 1.5*np.sin(2 * x) #+ 0.05 * (x-2)**1.2 #+ 0.02 * rng.normal(size=x.shape)
plt.figure()
plt.plot(x, y1, label='y1')
plt.plot(x, y2, label='y2')
plt.legend()
plt.show()

min_len = min(len(y1), len(y2))
y1, y2  = y1[:min_len], y2[:min_len]

time_delay, lag_dim = 20, 4
y1_delay_embeddings   = make_timedelay_embeddings(y1, time_delay, lag_dim)
y1_now, y1_delayed    = y1_delay_embeddings[:, 0], y1_delay_embeddings[:, 1]

# time_delay2, lag_dim2 = 20, 4
y2_delay_embeddings   = make_timedelay_embeddings(y2, time_delay, lag_dim)
y2_now, y2_delayed    = y2_delay_embeddings[:, 0], y2_delay_embeddings[:, 1]
plt.figure()
# plt.scatter(y1_now, y1_delayed, s=4)
plt.scatter(y2_now, y2_delayed, s=4)
plt.show()

# persistence stuff
persistence  = PersistenceAnalysis(max_dim=1)
n            = min(len(y1_delay_embeddings), len(y2_delay_embeddings))
y_all        = np.hstack((y1_delay_embeddings[:n], y2_delay_embeddings[:n]))
# y_data       = np.column_stack((y1_now, y1_delayed))
diagrams_list= persistence.plot_persistence_diagrams(y_all, to_plot=True)


In [ ]:
"TopoPersist algo"
num_timesteps = 3000
timesteps     = np.linspace(0, 14*np.pi, num_timesteps)
X1 = (0.5+0.5*timesteps) * np.sin(2*timesteps + 1) + 0.04 * rng.normal(size=timesteps.shape)
X2 = np.cos(3*timesteps + 1) + 0.04 * rng.normal(size=timesteps.shape)
X  = np.column_stack((X1, X2))

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

def window_sequence(sequence, window_size: int, stride: int = 1):
    """stride = window_size means no overlap, stride = 1 means maximum overlap"""
    if type(sequence) == np.ndarray:
        return np.array([sequence[i:i+window_size] for i in range(0, len(sequence) - window_size + 1, stride)])
    elif type(sequence) == torch.Tensor:
        return torch.stack([sequence[i:i+window_size] for i in range(0, len(sequence) - window_size + 1, stride)])

window_size = 100
stride      = window_size//5
X_windows   = window_sequence(X_scaled, window_size, stride)
X_train, X_test = train_test_split(X_windows, test_size=0.2, random_state=42, shuffle=False)

print("X shape", X.shape)
print("X_windows shape:", X_windows.shape, ", each window shape:", X_windows.shape[1:])
print("X_train/test shape:", X_train.shape, X_test.shape)


In [ ]:
"define + train LSTM encoder"
from torch.utils.data import DataLoader, TensorDataset

class LSTMAutoencoder(nn.Module):
    """LSTM autoencoder for time-series reconstruction + latent embedding. Good accuracy comes from
    teacher forcing in "decode": feeding the true previous value at each step instead of the predicted one"""

    def __init__(self, input_dim, latent_dim, encoder_hidden_dim=64, decoder_hidden_dim=64, epochs=100, learning_rate=0.001):
        """input_dim = #features"""
        super().__init__()

        self.device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.input_dim  = input_dim
        self.latent_dim = latent_dim
        self.epochs     = epochs
        self.learning_rate      = learning_rate
        self.encoder_hidden_dim = encoder_hidden_dim
        self.decoder_hidden_dim = decoder_hidden_dim

        # encoder: sequence -> hidden
        self.encoder   = nn.LSTM(input_size=input_dim, hidden_size=encoder_hidden_dim, batch_first=True)
        # self.decoder   = nn.LSTM(input_size=latent_dim, hidden_size=decoder_hidden_dim, batch_first=True)
        self.decoder   = nn.LSTM(input_size=input_dim + latent_dim, hidden_size=decoder_hidden_dim, batch_first=True)
        self.to_latent = nn.Linear(encoder_hidden_dim, latent_dim)

        # latent -> decoder initial state
        self.to_h0 = nn.Linear(latent_dim, decoder_hidden_dim)
        self.to_c0 = nn.Linear(latent_dim, decoder_hidden_dim)

        # output projection
        self.output_layer = nn.Linear(decoder_hidden_dim, input_dim)

    def encode(self, x):
        _, (h_n, _) = self.encoder(x)
        h = h_n[-1]
        z = self.to_latent(h)
        return z

    def decode(self, z, x):
        """LSTM decoder with teacher forcing: predicts x_t from (x_{t-1}, z) by feeding a right-shifted x concatenated with z at each step."""
        n_samples, n_rows, n_cols = x.size() # B = batch size, T = sequence length, D = input_dim
        h0 = self.to_h0(z).unsqueeze(0)
        c0 = self.to_c0(z).unsqueeze(0)

        # decoder_input = z.unsqueeze(1).repeat(1, seq_len, 1)

        # shift input right
        x_shift = torch.zeros_like(x)
        x_shift[:, 1:, :] = x[:, :-1, :]

        # concat latent at each step
        z_rep         = z.unsqueeze(1).expand(-1, n_rows, -1)
        decoder_input = torch.cat([x_shift, z_rep], dim=-1)

        y, _ = self.decoder(decoder_input, (h0, c0))
        out  = self.output_layer(y)
        return out

    def forward(self, x):
        x     = x.to(self.device)
        z     = self.encode(x)
        x_hat = self.decode(z, x)
        return x_hat, z

    def train_model(self, X, num_epochs=None, lr=None, patience=5, batch_size=32, device=None):
        device = device or self.device
        self.to(device)
        X = X.to(device)

        loader    = DataLoader(TensorDataset(X), batch_size=batch_size, shuffle=True)
        optimizer = torch.optim.AdamW(self.parameters(), lr=lr or self.learning_rate)
        loss_fn   = nn.MSELoss()
        losses    = []
        epochs    = num_epochs or self.epochs

        for epoch in range(epochs):
            epoch_loss = 0

            for (x_batch,) in loader:
                x_b = x_batch.to(device)
                optimizer.zero_grad()

                x_hat, _ = self.forward(x_b)
                loss     = loss_fn(x_hat, x_b)

                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()

            epoch_loss /= len(loader)
            losses.append(epoch_loss)

            if (epoch + 1) % 20 == 0:
                print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}")

            if len(losses) > patience and losses[-1] > losses[-patience]:
                print(f"Early stopping at epoch {epoch+1}")
                break
        return losses

def temporal_alignment_loss(z: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """Align latent and input velocity directions."""
    dz = z[:, 1:] - z[:, :-1]
    dx = x[:, 1:] - x[:, :-1]
    return ((dz.norm(dim=-1) - dx.norm(dim=-1))**2).mean()

def numpy_to_torch(X: np.ndarray) -> torch.Tensor:
    if isinstance(X, np.ndarray):
        X = torch.from_numpy(X).float()
    return X

X_train, X_test    = numpy_to_torch(X_train), numpy_to_torch(X_test)
input_dim          = X_train.shape[-1]
latent_dim         = 24
encoder_hidden_dim = 64
decoder_hidden_dim = 64
lstm_ae            = LSTMAutoencoder(input_dim, latent_dim, encoder_hidden_dim=encoder_hidden_dim,
                             decoder_hidden_dim=decoder_hidden_dim, epochs=100, learning_rate=0.001)
losses             = lstm_ae.train_model(X_train, patience=10)
lstm_ae.eval()
device             = lstm_ae.device

with torch.no_grad():
    x_hat, z_train = lstm_ae(X_train.to(device))

window_number = 8
plt.plot(x_hat[window_number, :, 0].cpu(), label='x1_hat')
plt.plot(X_train[window_number, :, 0].cpu(), label='x1')
plt.legend()
plt.show()

In [ ]:
window_size = z_train.shape[0]//2
stride      = window_size//8
z_train_windows = window_sequence(z_train, window_size, stride)
print(z_train_windows.shape, z_train.shape)

plt.figure()
plt.plot(z_train_windows[0, :, 0].cpu(), label='z1_win')
plt.show()
# ============

dataset        = z_train_windows[0] # z sample/"page"
persistence    = PersistenceAnalysis(max_dim=1)
diagrams_list  = persistence.compute_persistence_diagrams(dataset)
# diagrams_clean = persistence.remove_inf(diagrams_list)
# entropy_array  = persistence.compute_entropy(diagrams_clean)
# persistence_images_list = persistence.compute_persistence_image(diagrams_clean)
# betti_curves_array      = persistence.compute_betti_curves(diagrams_clean)

# print("Persistence array:", entropy_array)
# print("Betti curves shape: (batch, homology_dim, filtration_steps) =", betti_curves_array.shape)

# persistence.plot_persistence_diagrams(diagrams_list)
# persistence.plot_entropy(entropy_array)
# persistence.plot_persistence_image(persistence_images_list[1])  # H1
# persistence.plot_betti_curves(betti_curves_array)

def plot_multi_persistence_diagrams(z_train_windows, title: str, page_range: list = None):
    """z_train_windows shape: (num_windows, window_size, latent_dim)"""
    if page_range is None:
        page_range = range(z_train_windows.shape[0])

    for page in range(page_range[0], page_range[1]):
        dataset       = z_train_windows[page]
        persistence   = PersistenceAnalysis(max_dim=1)
        diagrams_list = persistence.compute_persistence_diagrams(dataset)
        persistence.plot_persistence_diagrams(diagrams_list, title=f"{title} {page}")

plot_multi_persistence_diagrams(z_train_windows, title="Window", page_range=[2,8])


In [ ]:
type(betti_curves_array)
# len(diagrams_clean)
